# Microestructura de Mercado — Ejercicios

Ejercicios guiados de Limit Order Book con datos de BTCUSDT.

## Idea pedagogica

- cargar y explorar datos reales de un order book
- calcular metricas basicas: spread, mid price, imbalance
- visualizar la estructura y dinamica del LOB

## Como usar este cuaderno

- **Nucleo de clase:** ejercicios 1 a 5
- **Si vamos bien:** ejercicios 6 y 7
- **Bonus / casa:** ejercicios 8 a 10

Intentalo primero. Mira la validacion despues. La solucion guiada existe para comparar enfoque, no para copiar sin pensar.

## 1. Carga los datos

**Practicas:** `pd.read_csv`, explorar un DataFrame.

Carga `../data/btc_lob_snapshots.csv` en un DataFrame llamado `df`. Imprime cuantas filas y columnas tiene.

In [ ]:
# Escribe aqui


In [ ]:
if "df" not in globals():
    print("Te falta crear `df` con pd.read_csv().")
elif len(df) != 500:
    print(f"El DataFrame tiene {len(df)} filas, se esperan 500.")
else:
    print(f"Bien: {len(df)} filas, {len(df.columns)} columnas.")

### Solucion guiada

```python
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("../data/btc_lob_snapshots.csv")
print(f"filas: {len(df)}, columnas: {len(df.columns)}")
```

## 2. Best bid y best ask

**Practicas:** acceder a columnas de un DataFrame.

Crea dos variables: `best_bid` y `best_ask` con las columnas `bid_price_1` y `ask_price_1` del primer snapshot (fila 0).

**Pista:** usa `df.iloc[0]` para acceder a la primera fila.

In [ ]:
# Escribe aqui


In [ ]:
missing = [v for v in ["best_bid", "best_ask"] if v not in globals()]
if missing:
    print("Te faltan:", missing)
elif best_bid >= best_ask:
    print(f"Revisa: best_bid ({best_bid}) deberia ser menor que best_ask ({best_ask}).")
else:
    print(f"Bien: best_bid=${best_bid:,.2f}, best_ask=${best_ask:,.2f}")

### Solucion guiada

```python
row = df.iloc[0]
best_bid = row["bid_price_1"]
best_ask = row["ask_price_1"]
print(f"best_bid=${best_bid:,.2f}, best_ask=${best_ask:,.2f}")
```

## 3. Spread y mid price

**Practicas:** operar sobre columnas.

Crea dos nuevas columnas en `df`:

- `df["spread"]` = `ask_price_1 - bid_price_1`
- `df["mid"]` = `(bid_price_1 + ask_price_1) / 2`

Imprime el spread y mid price medios.

In [ ]:
# Escribe aqui


In [ ]:
if "spread" not in df.columns or "mid" not in df.columns:
    print("Te faltan las columnas 'spread' y/o 'mid' en df.")
elif df["spread"].mean() < 1 or df["spread"].mean() > 100:
    print(f"Revisa: spread medio = {df['spread'].mean():.2f} — parece fuera de rango.")
else:
    print(f"Bien: spread medio = ${df['spread'].mean():.2f}, mid medio = ${df['mid'].mean():,.2f}")

### Solucion guiada

```python
df["spread"] = df["ask_price_1"] - df["bid_price_1"]
df["mid"] = (df["bid_price_1"] + df["ask_price_1"]) / 2

print(f"spread medio: ${df['spread'].mean():.2f}")
print(f"mid medio: ${df['mid'].mean():,.2f}")
```

## 4. Volumen bid vs ask

**Practicas:** sumar columnas con un bucle.

Crea dos nuevas columnas:

- `df["bid_vol_5"]` = suma de `bid_size_1` a `bid_size_5`
- `df["ask_vol_5"]` = suma de `ask_size_1` a `ask_size_5`

**Pista:** usa `sum(df[f"bid_size_{i}"] for i in range(1, 6))`.

In [ ]:
# Escribe aqui


In [ ]:
if "bid_vol_5" not in df.columns or "ask_vol_5" not in df.columns:
    print("Te faltan las columnas 'bid_vol_5' y/o 'ask_vol_5'.")
else:
    print(f"Bien: bid_vol_5 medio = {df['bid_vol_5'].mean():.2f} BTC")
    print(f"      ask_vol_5 medio = {df['ask_vol_5'].mean():.2f} BTC")

### Solucion guiada

```python
df["bid_vol_5"] = sum(df[f"bid_size_{i}"] for i in range(1, 6))
df["ask_vol_5"] = sum(df[f"ask_size_{i}"] for i in range(1, 6))

print(f"bid_vol_5 medio: {df['bid_vol_5'].mean():.2f} BTC")
print(f"ask_vol_5 medio: {df['ask_vol_5'].mean():.2f} BTC")
```

## 5. Calcula el imbalance

**Practicas:** formula con columnas.

Crea `df["imbalance"]` usando la formula:

```
imbalance = bid_vol_5 / (bid_vol_5 + ask_vol_5)
```

Imprime el imbalance medio y responde: en promedio, hay mas presion compradora o vendedora?

In [ ]:
# Escribe aqui


In [ ]:
if "imbalance" not in df.columns:
    print("Te falta la columna 'imbalance'.")
elif df["imbalance"].min() < 0 or df["imbalance"].max() > 1:
    print("Revisa: el imbalance deberia estar entre 0 y 1.")
else:
    mean_imb = df["imbalance"].mean()
    print(f"Bien: imbalance medio = {mean_imb:.4f}")
    if mean_imb > 0.5:
        print("→ En promedio, ligera presion compradora.")
    else:
        print("→ En promedio, ligera presion vendedora.")

### Solucion guiada

```python
df["imbalance"] = df["bid_vol_5"] / (df["bid_vol_5"] + df["ask_vol_5"])
print(f"imbalance medio: {df['imbalance'].mean():.4f}")
```

## 6. Visualiza un snapshot del LOB

**Practicas:** matplotlib barras horizontales.

Pinta el snapshot 0 como un grafico de barras horizontales:
- Bids en verde, con volumen hacia la izquierda (valores negativos)
- Asks en rojo, con volumen hacia la derecha

Usa los 10 niveles de cada lado.

**Pista:** usa `ax.barh(prices, sizes)` y pon los sizes de bid como negativos.

In [ ]:
# Escribe aqui


### Solucion guiada

```python
row = df.iloc[0]

bid_prices = [row[f"bid_price_{i}"] for i in range(1, 11)]
bid_sizes  = [row[f"bid_size_{i}"]  for i in range(1, 11)]
ask_prices = [row[f"ask_price_{i}"] for i in range(1, 11)]
ask_sizes  = [row[f"ask_size_{i}"]  for i in range(1, 11)]

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(bid_prices, [-s for s in bid_sizes], height=8, color="#4ade80", alpha=0.8, label="Bids")
ax.barh(ask_prices, ask_sizes, height=8, color="#f87171", alpha=0.8, label="Asks")
ax.set_xlabel("Volumen (BTC)")
ax.set_ylabel("Precio (USD)")
ax.set_title("Snapshot del LOB — BTCUSDT")
ax.legend()
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()
```

## 7. Imbalance en el tiempo

**Practicas:** grafico de linea con referencia.

Crea un eje de tiempo en minutos: `df["minutes"] = (df["timestamp"] - df["timestamp"].iloc[0]) / 60`.

Pinta `df["imbalance"]` a lo largo del tiempo con una linea horizontal en 0.5 como referencia de equilibrio.

In [ ]:
# Escribe aqui


### Solucion guiada

```python
df["minutes"] = (df["timestamp"] - df["timestamp"].iloc[0]) / 60

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(df["minutes"], df["imbalance"], color="#fbbf24", linewidth=0.8, alpha=0.7)
ax.axhline(y=0.5, color="#a1a1aa", linestyle="--", linewidth=0.8, label="Equilibrio")
ax.fill_between(df["minutes"], 0.5, df["imbalance"],
                where=df["imbalance"] > 0.5, color="#4ade80", alpha=0.2)
ax.fill_between(df["minutes"], 0.5, df["imbalance"],
                where=df["imbalance"] < 0.5, color="#f87171", alpha=0.2)
ax.set_xlabel("Minutos")
ax.set_ylabel("Imbalance")
ax.set_ylim(0, 1)
ax.set_title("Imbalance a lo largo del tiempo — BTCUSDT")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()
```

## 8. Depth chart

**Practicas:** volumen acumulado, `fill_between`.

Pinta un depth chart para el snapshot 0:
- Eje X = precio, eje Y = volumen acumulado
- Bids en verde (acumulando de nivel 1 a 10), asks en rojo
- Usa `ax.step()` y `ax.fill_between()` para el area

**Pista:** acumula los sizes con un bucle: para cada nivel, sumale al total anterior.

In [ ]:
# Escribe aqui


### Solucion guiada

```python
row = df.iloc[0]

bid_prices = [row[f"bid_price_{i}"] for i in range(1, 11)]
bid_sizes  = [row[f"bid_size_{i}"]  for i in range(1, 11)]
ask_prices = [row[f"ask_price_{i}"] for i in range(1, 11)]
ask_sizes  = [row[f"ask_size_{i}"]  for i in range(1, 11)]

bid_cum, ask_cum = [], []
total = 0
for s in bid_sizes:
    total += s
    bid_cum.append(total)
total = 0
for s in ask_sizes:
    total += s
    ask_cum.append(total)

fig, ax = plt.subplots(figsize=(10, 5))
ax.fill_between(bid_prices, bid_cum, alpha=0.3, color="#4ade80", step="mid")
ax.step(bid_prices, bid_cum, color="#4ade80", linewidth=2, where="mid", label="Bids")
ax.fill_between(ask_prices, ask_cum, alpha=0.3, color="#f87171", step="mid")
ax.step(ask_prices, ask_cum, color="#f87171", linewidth=2, where="mid", label="Asks")
ax.set_xlabel("Precio (USD)")
ax.set_ylabel("Volumen acumulado (BTC)")
ax.set_title("Depth Chart — BTCUSDT")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()
```

## 9. Imbalance vs cambio de precio

**Practicas:** correlacion, scatter plot.

Calcula el cambio del mid price entre snapshots consecutivos: `df["mid_change"] = df["mid"].diff()`.

Haz un scatter plot de `imbalance` (eje X) vs `mid_change` del snapshot siguiente (eje Y). Si el imbalance es predictivo, deberia haber una relacion positiva: imbalance alto → precio sube.

**Pista:** para alinear, usa `df["mid_change"].shift(-1)` para obtener el cambio *futuro*.

In [ ]:
# Escribe aqui


### Solucion guiada

```python
df["mid_change"] = df["mid"].diff()
df["future_change"] = df["mid_change"].shift(-1)

# Quitar NaN
valid = df.dropna(subset=["imbalance", "future_change"])

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(valid["imbalance"], valid["future_change"], alpha=0.3, s=10, color="#818cf8")
ax.axhline(y=0, color="#a1a1aa", linewidth=0.5)
ax.axvline(x=0.5, color="#a1a1aa", linewidth=0.5)
ax.set_xlabel("Imbalance")
ax.set_ylabel("Cambio futuro del mid (USD)")
ax.set_title("Imbalance vs cambio futuro del precio")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

corr = valid["imbalance"].corr(valid["future_change"])
print(f"Correlacion: {corr:.4f}")
if abs(corr) < 0.1:
    print("→ Correlacion debil: el imbalance informa, pero no predice con certeza.")
else:
    print(f"→ Correlacion {'positiva' if corr > 0 else 'negativa'}: senal de presion.")
```

## 10. Weighted mid price

**Practicas:** formula ponderada, comparacion de metricas.

El mid price clasico trata ambos lados como iguales. El **weighted mid** pondera por volumen del mejor nivel:

```
wmid = (bid_price_1 * ask_size_1 + ask_price_1 * bid_size_1) / (bid_size_1 + ask_size_1)
```

Calcula `df["wmid"]` y pinta la diferencia `wmid - mid` a lo largo del tiempo. Cuando es positiva, la presion compradora empuja el precio justo hacia arriba.

In [ ]:
# Escribe aqui


In [ ]:
if "wmid" not in df.columns:
    print("Te falta la columna 'wmid'.")
else:
    diff = (df["wmid"] - df["mid"]).abs().mean()
    print(f"Bien: diferencia media |wmid - mid| = ${diff:.4f}")
    if diff < 0.01:
        print("Revisa: la diferencia es muy pequena, puede haber un error.")
    else:
        print("El weighted mid captura la asimetria del libro.")

### Solucion guiada

```python
bp = df["bid_price_1"]
ap = df["ask_price_1"]
bs = df["bid_size_1"]
as_ = df["ask_size_1"]

df["wmid"] = (bp * as_ + ap * bs) / (bs + as_)

diff = df["wmid"] - df["mid"]

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(df["minutes"], diff, width=0.8, color="#818cf8", alpha=0.6)
ax.axhline(y=0, color="#a1a1aa", linewidth=0.5)
ax.set_xlabel("Minutos")
ax.set_ylabel("wmid - mid (USD)")
ax.set_title("Weighted mid vs mid: positivo = presion compradora")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Diferencia media: ${diff.mean():.4f}")
print(f"Diferencia max absoluta: ${diff.abs().max():.4f}")
```

## Cierre

Ya no trabajas con precios como datos fijos.

- Si llegas al ejercicio 5, ya sabes cargar un LOB, calcular spread, mid price e imbalance.
- Si llegas al 7, ya visualizas la estructura del libro y la dinamica del imbalance.
- Si llegas al 10, ya comparas metricas y entiendes la relacion entre presion y precio.

**Siguiente clase:** ya entiendes la estructura del LOB. En Lesson 5 veremos que pasa cuando envias una orden — tipos de ordenes (market, limit, stop) y como el matching engine las ejecuta.